# NLI Model Validation on Indonesian Data (IndoNLI)

This notebook evaluates the accuracy of the `mDeBERTa-v3-base-mnli-xnli` model on the IndoNLI dataset to demonstrate that the NLI model used is reliable for Indonesian text.

## 1. Check GPU & Install

In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

!curl -s --max-time 5 https://huggingface.co > /dev/null && echo "Internet: ON" || echo "Internet: OFF"

!pip install -q transformers datasets scikit-learn

GPU available: True
GPU name: Tesla T4
Internet: ON


## 2. Load IndoNLI Dataset

In [ ]:
import json
import os
from pathlib import Path

# Load IndoNLI from a local Kaggle dataset (offline)
BASE = "./data/indonli"

# Check folder contents
print("=== Dataset contents ===")
for root, dirs, files in os.walk(BASE):
    for f in files:
        print(os.path.join(root, f))

dataset = {}
for split in ["test_lay", "test_expert"]:
    filepath = None
    for root, dirs, files in os.walk(BASE):
        for f in files:
            if f == f"{split}.jsonl":
                filepath = os.path.join(root, f)

    if filepath is None:
        print(f"{split}: NOT FOUND")
        continue

    rows = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    dataset[split] = rows
    print(f"{split}: {len(rows)} samples")

print("\nSample data:")
print(dataset["test_lay"][0])

## 3. Load NLI Model

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

NLI_MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}
print("Model labels:", id2label)

ent_idx = next(i for i, l in id2label.items() if "entail" in l)
con_idx = next(i for i, l in id2label.items() if "contrad" in l)
neu_idx = next(i for i, l in id2label.items() if "neutral" in l)
print(f"Indices: entailment={ent_idx}, contradiction={con_idx}, neutral={neu_idx}")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model labels: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
Indices: entailment=0, contradiction=2, neutral=1


## 4. Evaluate on IndoNLI Test Sets

In [4]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import json

# IndoNLI label mapping: e=entailment, n=neutral, c=contradiction
LABEL_MAP = {"e": ent_idx, "n": neu_idx, "c": con_idx}
IDX_TO_NAME = {ent_idx: "entailment", neu_idx: "neutral", con_idx: "contradiction"}

def evaluate_split(split_data, split_name):
    y_true, y_pred = [], []

    for i, row in enumerate(split_data):
        premise = row["premise"]
        hypothesis = row["hypothesis"]
        gold_label = LABEL_MAP[row["label"]]

        with torch.inference_mode():
            enc = tokenizer(premise, hypothesis, truncation=True, max_length=512, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            logits = model(**enc).logits[0]
            pred_label = int(torch.argmax(logits).item())

        y_true.append(gold_label)
        y_pred.append(pred_label)

        if (i + 1) % 500 == 0:
            print(f"  [{split_name}] {i+1}/{len(split_data)}", flush=True)

    acc = accuracy_score(y_true, y_pred)
    target_names = [IDX_TO_NAME[i] for i in sorted(IDX_TO_NAME.keys())]
    report = classification_report(y_true, y_pred, target_names=target_names, digits=4)
    cm = confusion_matrix(y_true, y_pred)

    print(f"\n=== {split_name} ({len(split_data)} samples) ===")
    print(f"Accuracy: {acc:.4f}")
    print(report)
    print("Confusion Matrix:")
    print(cm)

    return {"split": split_name, "accuracy": acc, "samples": len(split_data)}

results = []

for split_name, split_data in dataset.items():
    result = evaluate_split(split_data, split_name)
    results.append(result)

  [test_lay] 500/2201
  [test_lay] 1000/2201
  [test_lay] 1500/2201
  [test_lay] 2000/2201

=== test_lay (2201 samples) ===
Accuracy: 0.7828
               precision    recall  f1-score   support

   entailment     0.7231    0.9245    0.8115       808
      neutral     0.8782    0.5390    0.6680       629
contradiction     0.8146    0.8338    0.8241       764

     accuracy                         0.7828      2201
    macro avg     0.8053    0.7657    0.7679      2201
 weighted avg     0.7992    0.7828    0.7749      2201

Confusion Matrix:
[[747  30  31]
 [176 339 114]
 [110  17 637]]
  [test_expert] 500/2984
  [test_expert] 1000/2984
  [test_expert] 1500/2984
  [test_expert] 2000/2984
  [test_expert] 2500/2984

=== test_expert (2984 samples) ===
Accuracy: 0.7590
               precision    recall  f1-score   support

   entailment     0.7019    0.8348    0.7626      1041
      neutral     0.7866    0.7733    0.7799       944
contradiction     0.8142    0.6667    0.7331       999

   

## 5. Results Summary

In [ ]:
from pathlib import Path

print("=== NLI Model Validation Summary ===")
print(f"Model: {NLI_MODEL_NAME}")
print(f"Dataset: IndoNLI")
print()
print(f"{'Split':<15} | {'Samples':>8} | {'Accuracy':>10}")
print("-" * 40)
for r in results:
    print(f"{r['split']:<15} | {r['samples']:>8} | {r['accuracy']:>10.4f}")

# Save results
output_path = Path("./results/nli_validation_results.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"\nSaved to {output_path}")